In [ ]:
import ipaddress
import pandas as pd
import matplotlib.pyplot as plt

# Keep consistent with other analysis notebooks
TIME_OFFSET = 10800

In [ ]:
def plot_sum_is_duplicate_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    time_offset_seconds=TIME_OFFSET,
):
    """
    1) Plot sum(is_duplicate) per src IP in 1-second bins.

    is_duplicate definition:
    df["is_duplicate"] = df.duplicated(
        subset=["ip.src", "ip.dst", "mbtcp.trans_id", "modbus.data"],
        keep=False
    )
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {
        "ip.src",
        "ip.dst",
        "frame.time_epoch",
        "mbtcp.trans_id",
        "modbus.data",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    # Exact duplicate rule requested by user.
    df["is_duplicate"] = df.duplicated(
        subset=["ip.src", "ip.dst", "mbtcp.trans_id", "modbus.data"],
        keep=False,
    )
    df["is_duplicate"] = df["is_duplicate"].astype(int)

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")["is_duplicate"].sum().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("sum(is_duplicate)")
    plt.title(f"Duplicate packet sum for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_trans_id_uniqueness_ratio_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    trans_id_col="mbtcp.trans_id",
    time_offset_seconds=TIME_OFFSET,
):
    """
    2) Plot nunique(mbtcp.trans_id) / packet_count per src IP in 1-second bins.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", trans_id_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    packet_count = window_df.groupby("second_bin").size().reindex(bins, fill_value=0)
    unique_trans_id = window_df.groupby("second_bin")[trans_id_col].nunique().reindex(bins, fill_value=0)
    ratio = (unique_trans_id / packet_count.where(packet_count != 0)).fillna(0.0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, ratio.values, marker="o", label=f"nunique({trans_id_col}) / packet_count")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("ratio")
    plt.title(f"Trans ID uniqueness ratio for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return ratio


def plot_mean_response_time_per_src_dst_pair(
    input_csv,
    src_ip,
    dst_ip,
    center_timestamp,
    interval_seconds,
    response_col="modbus.response_time",
    time_offset_seconds=TIME_OFFSET,
):
    """
    3) Plot mean(modbus.response_time) per (src, dst) pair in 1-second bins.
    """
    ipaddress.ip_address(src_ip)
    ipaddress.ip_address(dst_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "ip.dst", "frame.time_epoch", response_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )
    df[response_col] = pd.to_numeric(df[response_col], errors="coerce")

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    pair_df = df[(df["ip.src"].astype(str) == src_ip) & (df["ip.dst"].astype(str) == dst_ip)].copy()
    window_df = pair_df[(pair_df["aligned_ts"] >= start_ts) & (pair_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[response_col].mean().reindex(bins)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"mean({response_col})")
    plt.title(f"Mean response time for src={src_ip}, dst={dst_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


# Example usage:
# input_csv = "../train/cscada_attack_ssw.csv"
# src_ip = "185.175.0.5"
# dst_ip = "185.175.0.8"
# center_timestamp = "2023-03-19 03:01:57.813"
# x = 20
#
# plot_sum_is_duplicate_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_trans_id_uniqueness_ratio_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_mean_response_time_per_src_dst_pair(input_csv, src_ip, dst_ip, center_timestamp, x)